# Multiple Regression on CSV Upload

## Objective
Train multiple linear regression using all available features from uploaded CSV.

## Google Colab Steps
1. Run the setup cell.
2. Execute the main cell.
3. Save a copy to your Google Drive if needed.


In [ ]:
!pip -q install numpy pandas matplotlib seaborn scikit-learn scipy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)
display(df.head())

target_col = df.columns[-1]
X = df.drop(columns=[target_col])
y = pd.to_numeric(df[target_col], errors="coerce")

valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", LinearRegression()),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

plt.figure(figsize=(7, 4))
plt.scatter(y_test, pred, alpha=0.7)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Multiple Regression: Actual vs Predicted")
plt.show()
